<a href="https://colab.research.google.com/github/mariamalh/Mariam-Alhroob/blob/main/Hw_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# PART 1
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 1) Loading the dataset
df = pd.read_csv('diabetes_data.csv')

# 2) Droping rows containing missing values
df = df.dropna()

# 3) Droping irrelevant columns
df = df.drop(columns=['id'], errors='ignore')

# 4) Separate target and features
y = df['class']
X = df.drop(columns=['class'])

# 5) Encoding categorical features
for col in X.select_dtypes(include='object').columns:
    X[col] = LabelEncoder().fit_transform(X[col])

# 6) Encoding the target, positive = 1, negative = 0
if y.dtype == 'object':
    y = LabelEncoder().fit_transform(y)

# 7) Scaling numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Shape:', X_scaled.shape, 'Classes:', set(y))
print()

# PART 2
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedKFold, cross_val_score
import numpy as np

# 1) classification algorithms
models = {
    'Linear Classifier': Perceptron(max_iter=1000, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Gaussian NB': GaussianNB(),
    'Neural Network': MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=42)
}

# 2) 10-fold CV repeated 100 times (1,000 train/eval rounds in total)
rkf = RepeatedKFold(n_splits=10, n_repeats=100, random_state=42)

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y, cv=rkf, scoring='accuracy', n_jobs=-1)

    results[name] = (scores.mean(), scores.std())
    print(f"{name:20s} mean:{scores.mean():.4f} std:{scores.std():.4f}")

# PART 3

from sklearn.feature_selection import SequentialFeatureSelector

reduced_rkf = RepeatedKFold(n_splits=10, n_repeats=10, random_state=42)

feature_names = X.columns.tolist()

for name, model in models.items():
    # Forward Selection
    sfs = SequentialFeatureSelector(
      estimator = model,
      n_features_to_select = 'auto',
      direction = 'forward',
      scoring = 'accuracy',
      cv = reduced_rkf,
      n_jobs=-1,
    )
    sfs.fit(X_scaled, y)

    X_subset = sfs.transform(X_scaled)
    scores = cross_val_score(model, X_subset, y, cv=rkf, scoring='accuracy', n_jobs=-1)


    selected = np.array(feature_names)[sfs.get_support()]
    print('Selected features:', list(selected))
    print(f"{name:20s} mean:{scores.mean():.4f} std:{scores.std():.4f}")

Shape: (520, 16) Classes: {np.int64(0), np.int64(1)}

Linear Classifier    mean:0.8892 std:0.0466
Logistic Regression  mean:0.9259 std:0.0344
KNN                  mean:0.9253 std:0.0355
Gaussian NB          mean:0.8896 std:0.0421
Neural Network       mean:0.9739 std:0.0227
Selected features: [np.str_('Gender'), np.str_('Polyuria'), np.str_('Polydipsia'), np.str_('weakness'), np.str_('Genital thrush'), np.str_('Itching'), np.str_('Irritability'), np.str_('delayed healing')]
Linear Classifier    mean:0.8842 std:0.0525
Selected features: [np.str_('Gender'), np.str_('Polyuria'), np.str_('Polydipsia'), np.str_('Polyphagia'), np.str_('delayed healing'), np.str_('partial paresis'), np.str_('Alopecia'), np.str_('Obesity')]
Logistic Regression  mean:0.9164 std:0.0364
Selected features: [np.str_('Age'), np.str_('Gender'), np.str_('Polyuria'), np.str_('Polydipsia'), np.str_('Genital thrush'), np.str_('Itching'), np.str_('Irritability'), np.str_('Alopecia')]
KNN                  mean:0.9456 std:0.